# MP3 Generation Notebook

- 게임 플레이가 불가능할 시 이 파일에서 하나의 음악을 생성해볼 수 있다.
- CVAE-GRU로 mood별 1분 음악을 한 번 생성한다.
- 생성된 tensor를 MIDI로 바꾸고 ambient 스타일을 적용해 MP3로 렌더링한다.
- 최종 MP3는 Google Drive의 `midivae/output` 폴더에 저장한다.
- 자동재생은 사용하지 않고, 결과 확인용 미리듣기 플레이어만 표시한다.


## 0. 설치

In [ ]:
!apt-get -qq update
!apt-get -qq install -y fluidsynth fluid-soundfont-gm ffmpeg
!pip -q install pretty_midi pedalboard soundfile
print("✅ 설치 완료")

## 1. 생성 모델과 기본 설정
모델 구조, mood 매핑, seed 설정, MIDI 변환에 필요한 기본 코드를 정의한다.  
여기서 mood와 몇번째 epoch pt model을 쓸지 바꿀 수 있다. (맨 밑 cell에서 함수 실행 시 인자를 넣는 방식으로도 가능)


In [ ]:
import os
import random
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import pretty_midi
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pretty_midi"])
    import pretty_midi

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Colab 환경이 아니므로 Google Drive 마운트를 건너뜁니다.")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 중인 디바이스: {DEVICE}")
DRIVE_ROOT = Path('/content/drive/MyDrive') if Path('/content/drive/MyDrive').exists() else Path.cwd()
# 모델 버전은 epoch 100부터 240까지 있다. 여기서 원하는 epoch 숫자로 바꿔주면 된다.
BEST_MODEL_PATH = DRIVE_ROOT  / 'midivae' / 'saved_model' / 'final_train'/'base_topgenre247'/'checkpoints'/'epoch_200.pth' 
DATA_DIR = DRIVE_ROOT / 'midivae' / 'data'
OUTPUT_DIR = DRIVE_ROOT / 'midivae' / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MOOD_MAP = {
    'angry': 0,
    'exciting': 1,
    'fear': 2,
    'funny': 3,
    'happy': 4,
    'lazy': 5,
    'magnificent': 6,
    'quiet': 7,
    'romantic': 8,
    'sad': 9,
    'warm': 10,
}

MOOD_GENRE_MAP = {
    'angry': 'rock',
    'exciting': 'pop',
    'fear': 'rock',
    'funny': 'pop',
    'happy': 'country',
    'lazy': 'jazz',
    'magnificent': 'classical',
    'quiet': 'classical',
    'romantic': 'country',
    'sad': 'classical',
    'warm': 'country',
}

# 생성에 사용할 mood, 길이, MIDI 변환값, seed 옵션을 모아 둔다.
@dataclass(frozen=True)
class GenerationConfig:
    target_mood: str = 'sad' # 여기서 mood를 바꿀 수 있다.
    generate_steps: int = 64
    bpm: int = 120
    threshold: float = 0.5
    velocity: int = 100
    temperature: float = 3
    seed: int | None = None

    # seed chunk를 뽑을 데이터셋 경로와 선택 옵션
    data_dir: Path = DATA_DIR
    seed_pt_path: str | None = None
    seed_chunk_index: int | None = None

    # True면 출력 앞부분에 seed chunk를 포함한다.
    include_seed_chunk_in_output: bool = False

CONFIG = GenerationConfig()


In [ ]:
# mood 조건을 받아 다음 piano-roll chunk를 생성하는 CVAE-GRU 모델이다.
class CVAE_GRU(nn.Module):
    # CNN 인코더, GRU 상태 갱신부, VAE 잠재공간, CNN 디코더를 정의한다.
    def __init__(self, num_moods=11, embed_dim=32, channels=4, feature_dim=256, hidden_dim=256, latent_dim=128):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        self.input_shape = (channels, 88, 32)
        self.cnn_shape = (128, 11, 4)
        self.flatten_dim = np.prod(self.cnn_shape).item()

        self.mood_embedding = nn.Embedding(num_moods, embed_dim)
        self.encoder_cnn = nn.Sequential(
            nn.Conv2d(channels, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        self.feature_proj = nn.Linear(self.flatten_dim, feature_dim)
        self.gru_cell = nn.GRUCell(input_size=feature_dim + embed_dim, hidden_size=hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        self.decoder_proj = nn.Linear(latent_dim + embed_dim + hidden_dim, self.flatten_dim)
        self.decoder_cnn = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=(1, 1)),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=(1, 1)),
            nn.ReLU(),
            nn.ConvTranspose2d(32, channels, kernel_size=3, stride=2, padding=1, output_padding=(1, 1)),
            nn.Sigmoid(),
        )

    # 입력 chunk와 mood를 hidden state로 압축하고 mu/logvar를 계산한다.
    def encode(self, x, mood_vector, prev_hidden):
        cnn_features = self.encoder_cnn(x)
        features = F.relu(self.feature_proj(cnn_features))
        gru_input = torch.cat([features, mood_vector], dim=1)
        current_hidden = self.gru_cell(gru_input, prev_hidden)
        mu = self.fc_mu(current_hidden)
        logvar = self.fc_logvar(current_hidden)
        return mu, logvar, current_hidden

    # mu/logvar에서 미분 가능한 방식으로 latent vector z를 샘플링한다.
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    # latent vector, mood, hidden state를 piano-roll chunk로 복원한다.
    def decode(self, z, mood_vector, current_hidden):
        decoder_input = torch.cat([z, mood_vector, current_hidden], dim=1)
        x = F.relu(self.decoder_proj(decoder_input))
        x = x.view(-1, *self.cnn_shape)
        decoded = self.decoder_cnn(x)
        return decoded[..., :self.input_shape[1], :self.input_shape[2]]

    # 한 시점의 chunk를 처리하고 복원값과 다음 hidden state를 반환한다.
    def forward(self, x, mood_id, prev_hidden=None):
        batch_size = x.size(0)
        if prev_hidden is None:
            prev_hidden = torch.zeros(batch_size, self.hidden_dim, device=x.device)

        mood_vector = self.mood_embedding(mood_id)
        mu, logvar, current_hidden = self.encode(x, mood_vector, prev_hidden)
        z = self.reparameterize(mu, logvar)
        reconstruction = self.decode(z, mood_vector, current_hidden)
        return reconstruction, mu, logvar, current_hidden

In [ ]:
# BCE 재구성 손실과 KL divergence를 합쳐 VAE loss를 계산한다.
def vae_loss_fn(recon_x, x, mu, logvar, beta=1.0):
    bce = F.binary_cross_entropy(recon_x, x, reduction='sum')
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    batch_size = x.size(0)
    total_loss = (bce + beta * kld) / batch_size
    return total_loss, bce / batch_size, kld / batch_size


# Python, NumPy, Torch 난수 시드를 고정하고 실제 사용된 seed를 반환한다.
def set_seed(seed):
    if seed is None:
        seed = int.from_bytes(os.urandom(4), "little")
        print("run seed:", seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    return seed


# 저장된 checkpoint를 읽어 생성용 CVAE-GRU 모델을 eval 모드(추론/생성용)로 준비한다.
def load_model(model_path, device):
    if not Path(model_path).exists():
        raise FileNotFoundError(f"모델 가중치 파일을 찾을 수 없습니다: {model_path}")

    model = CVAE_GRU(num_moods=len(MOOD_MAP), hidden_dim=512, latent_dim=256).to(device)
    checkpoint = torch.load(model_path, map_location=device)
    state_dict = checkpoint.get('model_state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint
    model.load_state_dict(state_dict)
    model.eval()
    return model


def _extract_tensor_from_pt(obj):
    """pt 파일 안에서 실제 chunk tensor를 찾아 꺼낸다."""
    if isinstance(obj, torch.Tensor):
        return obj

    if isinstance(obj, dict):
        # 자주 쓰는 key를 먼저 확인한다.
        preferred_keys = [
            'chunks', 'chunk', 'data', 'tensor', 'piano_roll', 'pianoroll',
            'x', 'input', 'inputs', 'melody', 'arr', 'array'
        ]
        for key in preferred_keys:
            if key in obj:
                try:
                    return _extract_tensor_from_pt(obj[key])
                except Exception:
                    pass

        # key를 모르면 가장 차원이 큰 tensor를 사용한다.
        candidates = []
        for value in obj.values():
            try:
                tensor = _extract_tensor_from_pt(value)
                if isinstance(tensor, torch.Tensor):
                    candidates.append(tensor)
            except Exception:
                pass
        if candidates:
            return max(candidates, key=lambda t: t.ndim)

    if isinstance(obj, (list, tuple)):
        candidates = []
        for value in obj:
            try:
                tensor = _extract_tensor_from_pt(value)
                if isinstance(tensor, torch.Tensor):
                    candidates.append(tensor)
            except Exception:
                pass
        if candidates:
            return max(candidates, key=lambda t: t.ndim)

    raise TypeError(f"pt 파일에서 tensor를 찾지 못했습니다. type={type(obj)}")


def _to_chunk_bank(tensor):
    """여러 shape의 tensor를 [N, 4, 88, 32] chunk 묶음으로 맞춘다."""
    tensor = torch.as_tensor(tensor).detach().cpu().float()

    # MIDI velocity처럼 큰 값이면 0~1 범위로 정규화한다.
    if tensor.numel() > 0 and tensor.max() > 1.5:
        tensor = tensor / float(tensor.max())

    # [4, 88, 32]
    if tensor.ndim == 3:
        if tuple(tensor.shape) == (4, 88, 32):
            return tensor.unsqueeze(0).contiguous()
        # [88, 32, 4]
        if tuple(tensor.shape) == (88, 32, 4):
            return tensor.permute(2, 0, 1).unsqueeze(0).contiguous()
        raise ValueError(f"지원하지 않는 3D tensor shape입니다: {tuple(tensor.shape)}")

    # [N, 4, 88, 32]
    if tensor.ndim == 4:
        if tensor.shape[1:] == (4, 88, 32):
            return tensor.contiguous()
        # [N, 88, 32, 4]
        if tensor.shape[1:] == (88, 32, 4):
            return tensor.permute(0, 3, 1, 2).contiguous()
        # [4, 88, 32, N] 같은 경우는 마지막 차원을 chunk 축으로 이동
        if tensor.shape[:3] == (4, 88, 32):
            return tensor.permute(3, 0, 1, 2).contiguous()
        raise ValueError(f"지원하지 않는 4D tensor shape입니다: {tuple(tensor.shape)}")

    # [song, N, 4, 88, 32] 또는 [batch, seq, 4, 88, 32]
    if tensor.ndim == 5:
        if tensor.shape[-3:] == (4, 88, 32):
            return tensor.reshape(-1, 4, 88, 32).contiguous()
        raise ValueError(f"지원하지 않는 5D tensor shape입니다: {tuple(tensor.shape)}")

    raise ValueError(f"지원하지 않는 tensor 차원입니다: ndim={tensor.ndim}, shape={tuple(tensor.shape)}")


def load_real_seed_chunk(data_dir, mood_name, device=DEVICE, seed_pt_path=None, seed_chunk_index=None):
    """현재 mood의 .pt 파일에서 실제 seed chunk 하나를 뽑아 모델 입력 형태로 반환한다."""
    if mood_name not in MOOD_MAP:
        valid_moods = ', '.join(MOOD_MAP)
        raise ValueError(f"알 수 없는 mood입니다: {mood_name}. 사용 가능: {valid_moods}")

    data_dir = Path(data_dir)

    # mood별 대표 genre 폴더에서 seed 파일을 찾는다.
    if mood_name not in MOOD_GENRE_MAP:
        raise ValueError(f"알 수 없는 mood입니다: {mood_name}. MOOD_GENRE_MAP에 추가해야 합니다.")
    genre_name = MOOD_GENRE_MAP[mood_name]
    mood_dir = data_dir / mood_name / genre_name

    if seed_pt_path is None:
        if not mood_dir.exists():
            raise FileNotFoundError(f"mood 폴더를 찾을 수 없습니다: {mood_dir}")
        pt_files = sorted(mood_dir.glob('*.pt'))
        if not pt_files:
            raise FileNotFoundError(f"{mood_dir} 안에서 .pt 파일을 찾지 못했습니다.")
        pt_path = random.choice(pt_files)
    else:
        pt_path = Path(seed_pt_path)
        if not pt_path.is_absolute():
            # 파일명만 넣으면 현재 mood/genre 폴더 기준으로 찾는다.
            pt_path = mood_dir / pt_path
        if not pt_path.exists():
            raise FileNotFoundError(f"지정한 seed pt 파일을 찾을 수 없습니다: {pt_path}")

    obj = torch.load(pt_path, map_location='cpu')
    tensor = _extract_tensor_from_pt(obj)
    chunk_bank = _to_chunk_bank(tensor)

    if len(chunk_bank) == 0:
        raise ValueError(f"선택된 pt 파일에 chunk가 없습니다: {pt_path}")

    if seed_chunk_index is None:
        chunk_idx = random.randrange(len(chunk_bank))
    else:
        chunk_idx = int(seed_chunk_index)
        if not (0 <= chunk_idx < len(chunk_bank)):
            raise IndexError(f"seed_chunk_index 범위 오류: {chunk_idx}, 가능 범위: 0~{len(chunk_bank)-1}")

    seed_chunk = chunk_bank[chunk_idx].clamp(0, 1).unsqueeze(0).to(device)
    info = {
        'pt_path': str(pt_path),
        'pt_name': pt_path.name,
        'chunk_index': chunk_idx,
        'num_chunks_in_pt': len(chunk_bank),
        'seed_shape': tuple(seed_chunk.shape),
    }
    return seed_chunk, info


# 실제 seed chunk에서 시작해 지정한 mood의 다음 chunk들을 순차 생성한다.
def generate_chunks(
    model,
    mood_name,
    steps=16,
    temperature=1.0,
    device=DEVICE,
    data_dir=DATA_DIR,
    seed_pt_path=None,
    seed_chunk_index=None,
    include_seed_chunk_in_output=False,
):
    if mood_name not in MOOD_MAP:
        valid_moods = ', '.join(MOOD_MAP)
        raise ValueError(f"알 수 없는 mood입니다: {mood_name}. 사용 가능: {valid_moods}")
    if steps <= 0:
        raise ValueError('generate_steps는 1 이상이어야 합니다.')
    if temperature <= 0:
        raise ValueError('temperature는 0보다 커야 합니다.')

    mood_id = torch.tensor([MOOD_MAP[mood_name]], dtype=torch.long, device=device)
    generated_chunks = []

    with torch.inference_mode():
        mood_vector = model.mood_embedding(mood_id)

        # 실제 dataset chunk를 시작점으로 사용한다.
        current_x, seed_info = load_real_seed_chunk(
            data_dir=data_dir,
            mood_name=mood_name,
            device=device,
            seed_pt_path=seed_pt_path,
            seed_chunk_index=seed_chunk_index,
        )

        # seed chunk를 한 번 encode해 hidden state를 초기화한다.
        hidden = torch.zeros(1, model.hidden_dim, device=device)
        _, _, hidden = model.encode(current_x, mood_vector, hidden)

        if include_seed_chunk_in_output:
            generated_chunks.append(current_x.squeeze(0).detach().cpu())

        for _ in range(steps):
            z = torch.randn(1, model.latent_dim, device=device) * temperature
            recon = model.decode(z, mood_vector, hidden)
            generated_chunks.append(recon.squeeze(0).detach().cpu())

            # 생성된 chunk를 다시 encode해 다음 step의 context로 쓴다.
            current_x = recon
            _, _, hidden = model.encode(current_x, mood_vector, hidden)

    generated = torch.stack(generated_chunks, dim=0)
    return generated, seed_info

In [ ]:
def tensor_to_piano_roll(generated_tensor, threshold=0.5):
    """생성 tensor를 [channels, 88, total_steps] piano roll로 변환한다."""
    if isinstance(generated_tensor, torch.Tensor):
        array = generated_tensor.detach().cpu().numpy()
    else:
        array = np.asarray(generated_tensor)

    if array.ndim == 5:
        if array.shape[0] != 1:
            raise ValueError(f"배치 크기 1만 지원합니다. 현재 shape: {array.shape}")
        array = array[0]
    if array.ndim != 4:
        raise ValueError(f"입력 shape는 [chunks, channels, 88, steps]이어야 합니다. 현재 shape: {array.shape}")

    binary = (array > threshold).astype(np.uint8)
    return np.concatenate(list(binary), axis=-1)


def tensor_to_midi(generated_tensor, output_filename, bpm=120, threshold=0.5, velocity=100):
    """생성 tensor를 MIDI 파일로 저장하고 piano roll을 반환한다."""
    full_piano_roll = tensor_to_piano_roll(generated_tensor, threshold=threshold)
    channels, pitches, total_steps = full_piano_roll.shape
    step_duration = (60.0 / bpm) / 4.0
    velocity = int(np.clip(velocity, 1, 127))

    instruments = [
        pretty_midi.Instrument(program=0, is_drum=False, name="Melody"),
        pretty_midi.Instrument(program=48, is_drum=False, name="Harmony"),
        pretty_midi.Instrument(program=32, is_drum=False, name="Bass"),
        pretty_midi.Instrument(program=0, is_drum=True, name="Drums"),
    ]

    pm = pretty_midi.PrettyMIDI(initial_tempo=bpm)
    for channel_idx in range(min(channels, len(instruments))):
        instrument = instruments[channel_idx]
        for pitch_idx in range(pitches):
            step = 0
            while step < total_steps:
                if full_piano_roll[channel_idx, pitch_idx, step] == 0:
                    step += 1
                    continue

                start_step = step
                while step < total_steps and full_piano_roll[channel_idx, pitch_idx, step] == 1:
                    step += 1
                end_step = step

                instrument.notes.append(
                    pretty_midi.Note(
                        velocity=velocity,
                        pitch=pitch_idx + 21,
                        start=start_step * step_duration,
                        end=end_step * step_duration,
                    )
                )

        if instrument.notes:
            pm.instruments.append(instrument)

    output_filename = Path(output_filename)
    output_filename.parent.mkdir(parents=True, exist_ok=True)
    pm.write(str(output_filename))
    print(f"MIDI 생성 완료: {output_filename}")
    return full_piano_roll


# channel별 piano roll을 합쳐 이미지로 시각화하고 선택적으로 저장한다.
def plot_piano_roll(piano_roll, save_path=None):
    piano_roll = np.asarray(piano_roll, dtype=np.float32)
    if piano_roll.ndim != 3:
        raise ValueError(f"piano_roll shape는 [channels, 88, total_steps]이어야 합니다. 현재 shape: {piano_roll.shape}")

    combined_roll = piano_roll.max(axis=0)
    fig, ax = plt.subplots(figsize=(15, 6))
    image = ax.imshow(combined_roll, aspect='auto', origin='lower', cmap='magma', interpolation='nearest')

    y_ticks = np.arange(0, 88, 12)
    y_labels = [pretty_midi.note_number_to_name(note + 21) for note in y_ticks]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)
    ax.set_xlabel('Time Steps (1 step = 1/16 note)')
    ax.set_ylabel('Pitch')
    ax.set_title('Generated MIDI Piano Roll')
    fig.colorbar(image, ax=ax, label='Note On')
    fig.tight_layout()

    if save_path:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=300)
        print(f"시각화 이미지 저장 완료: {save_path}")

    plt.show()
    return fig, ax

## 2. 1분 생성·Ambient 렌더링 설정

BPM=120에서 현재 모델 tensor의 한 chunk(`32`개의 16분음표 step)는 **4초**다.  
그래서 1분은 **15 chunks**로 생성하면 된다. 원본 노트북의 `generate_steps=64`는 약 256초였기 때문에 여기서는 사용하지 않는다.

In [ ]:
import base64
import json
import math
import time
from collections import Counter

from pedalboard import Pedalboard, Reverb, Delay, Chorus, Compressor, Gain, Limiter
from pedalboard.io import AudioFile
from IPython.display import Audio, display

# 1분 clip 길이와 chunk 개수 설정
BPM = 120
CLIP_SECONDS = 60.0
SECONDS_PER_CHUNK = 32 * (60.0 / BPM) / 4.0
CHUNKS_PER_CLIP = math.ceil(CLIP_SECONDS / SECONDS_PER_CHUNK)  # BPM=120이면 15

# CVAE-GRU 생성 파라미터
GEN_TEMPERATURE = 1.5
MIDI_THRESHOLD = 0.65
MIDI_VELOCITY = 105

# Ambient MIDI 변환 설정
AMBIENT_STYLE = "warm"  # new_age / warm / polysynth / choir / bowed / metallic / halo / sweep
AMBIENT_PROGRAMS = {
    "new_age": 88, "warm": 89, "polysynth": 90, "choir": 91,
    "bowed": 92, "metallic": 93, "halo": 94, "sweep": 95,
}
AMBIENT_PROGRAM = AMBIENT_PROGRAMS[AMBIENT_STYLE]

DENSITY_KEEP_PROB = 0.55       # 낮을수록 note를 더 많이 줄인다.
NOTE_LENGTH_MULTIPLIER = 2.4
VELOCITY_SCALE = 0.78
REMOVE_DRUMS = False
ADD_DRONE = True

# 1분 audio에 적용할 fade와 sample rate
FADE_IN_SECONDS = 2.0
FADE_OUT_SECONDS = 3.0
SAMPLE_RATE = 44100

# 여기 값을 바꾸거나 generate_single_mp3(mood_name="sad")처럼 호출해서 원하는 mood를 지정한다.
DEFAULT_START_MOOD = "quiet"

# 최종 MP3는 Google Drive output 폴더에 저장하고, 렌더링 중간 파일은 임시 폴더에 둔다.
FINAL_MP3_OUTPUT_DIR = OUTPUT_DIR
FINAL_MP3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RENDER_WORK_DIR = Path("/tmp/midivae_mp3_work")
RENDER_WORK_DIR.mkdir(parents=True, exist_ok=True)

assert CHUNKS_PER_CLIP == 15, (
    f"BPM={BPM}에서는 1분이 15 chunks여야 합니다. 현재: {CHUNKS_PER_CLIP}"
)

print(f"한 chunk = {SECONDS_PER_CHUNK:.2f}초")
print(f"1분 = {CHUNKS_PER_CLIP} chunks")
print(f"MP3 저장 폴더: {FINAL_MP3_OUTPUT_DIR}")
print(f"기본 생성 mood: {DEFAULT_START_MOOD}")


## 3. 1분 단위 생성
현재 mood 폴더에서 실제 dataset chunk를 seed로 뽑고, 그 seed로 GRU hidden을 초기화한 뒤 15개 chunk를 생성한다.


In [ ]:
@dataclass
class LiveGenerationState:
    """현재 segment와 마지막 seed 정보를 표시하기 위한 상태이다."""
    segment_index: int = 0
    last_seed_info: dict | None = None


def generate_one_minute_tensor(
    model,
    mood_name,
    chunks_per_clip=CHUNKS_PER_CLIP,
    temperature=GEN_TEMPERATURE,
    data_dir=DATA_DIR,
):
    """현재 mood의 seed chunk로 hidden을 초기화하고 1분 분량 tensor를 생성한다."""
    if mood_name not in MOOD_MAP:
        raise ValueError(f"알 수 없는 mood: {mood_name}")

    generated, seed_info = generate_chunks(
        model=model,
        mood_name=mood_name,
        steps=chunks_per_clip,
        temperature=temperature,
        device=DEVICE,
        data_dir=data_dir,
        include_seed_chunk_in_output=False,
    )

    state = LiveGenerationState(
        segment_index=0,
        last_seed_info=seed_info,
    )
    return generated, state


## 4. MIDI → Ambient audio + 정확히 60초 fade-in / fade-out
`midi_to_ambient_colab.ipynb`의 pad 변환과 FX를 함수로 합친 셀이다.

In [ ]:
# 값을 정수 범위 안으로 제한한다.
def _clip_int(value, low, high):
    return int(max(low, min(high, value)))


# MIDI note 분포에서 drone에 사용할 중심 pitch class를 추정한다.
def _estimate_tonic_pitch_class(pm):
    weights = Counter()
    for inst in pm.instruments:
        if inst.is_drum:
            continue
        for note in inst.notes:
            weights[note.pitch % 12] += max(0.01, note.end - note.start) * max(1, note.velocity)
    return weights.most_common(1)[0][0] if weights else 0


def transform_to_ambient(input_midi_path, output_midi_path):
    """원본 MIDI의 note 밀도를 줄이고 pad/drone 중심의 ambient MIDI로 변환한다."""
    src = pretty_midi.PrettyMIDI(str(input_midi_path))
    source_end = src.get_end_time()
    _, tempi = src.get_tempo_changes()
    tempo = float(tempi[0]) if len(tempi) else float(BPM)

    out = pretty_midi.PrettyMIDI(initial_tempo=tempo)

    for idx, src_inst in enumerate(src.instruments):
        if src_inst.is_drum and REMOVE_DRUMS:
            continue

        if src_inst.is_drum:
            dst = pretty_midi.Instrument(
                program=src_inst.program, is_drum=True, name=f"soft_drum_{idx}"
            )
        else:
            dst = pretty_midi.Instrument(
                program=AMBIENT_PROGRAM, is_drum=False, name=f"ambient_{AMBIENT_STYLE}_{idx}"
            )
            dst.control_changes.extend([
                pretty_midi.ControlChange(number=91, value=105, time=0.0),
                pretty_midi.ControlChange(number=93, value=72, time=0.0),
                pretty_midi.ControlChange(number=7, value=88, time=0.0),
            ])

        for note in src_inst.notes:
            if random.random() > DENSITY_KEEP_PROB:
                continue

            if src_inst.is_drum:
                new_note = pretty_midi.Note(
                    velocity=_clip_int(note.velocity * 0.35, 15, 50),
                    pitch=note.pitch, start=note.start, end=note.end,
                )
            else:
                duration = min(max(0.30, (note.end - note.start) * NOTE_LENGTH_MULTIPLIER), 8.0)
                new_note = pretty_midi.Note(
                    velocity=_clip_int(note.velocity * VELOCITY_SCALE, 24, 72),
                    pitch=note.pitch,
                    start=note.start,
                    end=min(note.start + duration, source_end + 3.0),
                )
            dst.notes.append(new_note)

        if dst.notes:
            dst.notes.sort(key=lambda n: (n.start, n.pitch))
            out.instruments.append(dst)

    if ADD_DRONE and source_end > 0:
        tonic = _estimate_tonic_pitch_class(src)
        root, fifth = 36 + tonic, 43 + tonic
        drone = pretty_midi.Instrument(program=AMBIENT_PROGRAM, is_drum=False, name="ambient_drone")
        drone.control_changes.extend([
            pretty_midi.ControlChange(number=91, value=115, time=0.0),
            pretty_midi.ControlChange(number=93, value=85, time=0.0),
            pretty_midi.ControlChange(number=7, value=72, time=0.0),
        ])
        t = 0.0
        while t < source_end:
            end = min(t + 12.0, source_end + 3.0)
            drone.notes.append(pretty_midi.Note(velocity=33, pitch=root, start=t, end=end))
            drone.notes.append(pretty_midi.Note(velocity=27, pitch=fifth, start=t, end=end))
            t += 8.0
        out.instruments.append(drone)

    output_midi_path = Path(output_midi_path)
    out.write(str(output_midi_path))
    return output_midi_path

FX_BOARD = Pedalboard([
    Chorus(rate_hz=0.30, depth=0.40, centre_delay_ms=15, feedback=0.10, mix=0.22),
    Reverb(room_size=0.88, damping=0.38, wet_level=0.36, dry_level=0.72),
    Delay(delay_seconds=0.45, feedback=0.20, mix=0.15),
    Compressor(threshold_db=-20, ratio=2.0),
    Gain(gain_db=3.0),
    Limiter(threshold_db=-1.0),
])

SF2_PATH = "/usr/share/sounds/sf2/FluidR3_GM.sf2"


def force_60_seconds_and_fade(audio, sample_rate):
    """audio를 정확히 60초 길이로 맞추고 fade-in/out을 적용한다."""
    target_frames = int(round(CLIP_SECONDS * sample_rate))
    channels, frames = audio.shape

    if frames >= target_frames:
        fixed = audio[:, :target_frames].copy()
    else:
        fixed = np.pad(audio, ((0, 0), (0, target_frames - frames)), mode="constant")

    fade_in_frames = min(int(FADE_IN_SECONDS * sample_rate), target_frames // 2)
    fade_out_frames = min(int(FADE_OUT_SECONDS * sample_rate), target_frames // 2)

    if fade_in_frames > 1:
        fade_in = np.sin(np.linspace(0, np.pi / 2, fade_in_frames)) ** 2
        fixed[:, :fade_in_frames] *= fade_in[None, :]

    if fade_out_frames > 1:
        fade_out = np.cos(np.linspace(0, np.pi / 2, fade_out_frames)) ** 2
        fixed[:, -fade_out_frames:] *= fade_out[None, :]

    peak = np.max(np.abs(fixed))
    if peak > 0.98:
        fixed *= 0.98 / peak
    return fixed


def render_segment_to_mp3(generated_tensor, segment_no, mood_name):
    """생성 tensor를 MIDI, ambient MIDI, WAV 효과 처리를 거쳐 60초 MP3로 저장한다."""
    start_time = time.perf_counter()
    stem = f"generated_{segment_no:04d}_{mood_name}_{int(time.time())}"

    raw_mid = RENDER_WORK_DIR / f"{stem}_raw.mid"
    ambient_mid = RENDER_WORK_DIR / f"{stem}_ambient.mid"
    raw_wav = RENDER_WORK_DIR / f"{stem}_raw.wav"
    final_wav = RENDER_WORK_DIR / f"{stem}_fade_fx.wav"
    final_mp3 = FINAL_MP3_OUTPUT_DIR / f"{stem}.mp3"

    tensor_to_midi(
        generated_tensor,
        raw_mid,
        bpm=BPM,
        threshold=MIDI_THRESHOLD,
        velocity=MIDI_VELOCITY,
    )
    transform_to_ambient(raw_mid, ambient_mid)

    subprocess.run([
        "fluidsynth", "-ni", SF2_PATH, str(ambient_mid),
        "-F", str(raw_wav), "-r", str(SAMPLE_RATE),
    ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    with AudioFile(str(raw_wav)) as audio_file:
        audio = audio_file.read(audio_file.frames)
        sr = audio_file.samplerate

    effected = FX_BOARD(audio, sr)
    finished = force_60_seconds_and_fade(effected, sr)

    with AudioFile(str(final_wav), "w", sr, finished.shape[0]) as audio_file:
        audio_file.write(finished)

    subprocess.run([
        "ffmpeg", "-y", "-i", str(final_wav),
        "-ar", str(sr), "-ac", str(finished.shape[0]),
        "-codec:a", "libmp3lame", "-b:a", "192k",
        str(final_mp3),
    ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    return final_mp3, round(time.perf_counter() - start_time, 2)


## 5. mood 직접 선택과 MP3 생성 함수
외부 파일을 읽지 않고, 함수에 넘긴 mood 값으로 바로 1분 MP3를 생성한다.


In [ ]:
# 지정한 mood로 이번 MP3를 생성한다.
MODEL = load_model(BEST_MODEL_PATH, DEVICE)

LAST_SEED_INFO = None
SEGMENT_NO = 0


# 입력 mood 문자열을 표준 형태로 정리하고 지원 여부를 확인한다.
def _normalize_mood(mood):
    mood = str(mood).strip().lower()
    if mood not in MOOD_MAP:
        raise ValueError(f"잘못된 mood: {mood}. 가능: {', '.join(MOOD_MAP)}")
    return mood


# seed 정보가 이전 생성과 같은지 비교하기 위한 key를 만든다.
def _seed_key(seed_info):
    if not seed_info:
        return None
    return (seed_info.get("pt_path"), seed_info.get("chunk_index"))


def build_one_mp3(mood_name=DEFAULT_START_MOOD):
    """지정한 mood로 1분 tensor 생성부터 MP3 저장까지 한 번만 수행한다."""
    global SEGMENT_NO, LAST_SEED_INFO

    mood_name = _normalize_mood(mood_name)
    previous_seed_info = LAST_SEED_INFO
    build_started_at = time.strftime("%Y-%m-%d %H:%M:%S")

    generated, state = generate_one_minute_tensor(
        MODEL,
        mood_name,
        chunks_per_clip=CHUNKS_PER_CLIP,
        temperature=GEN_TEMPERATURE,
        data_dir=DATA_DIR,
    )

    LAST_SEED_INFO = state.last_seed_info
    same_as_previous_seed = _seed_key(LAST_SEED_INFO) == _seed_key(previous_seed_info)
    SEGMENT_NO += 1

    mp3_path, elapsed = render_segment_to_mp3(generated, SEGMENT_NO, mood_name)

    genre_name = MOOD_GENRE_MAP.get(mood_name)
    expected_seed_dir = str(Path(DATA_DIR) / mood_name / str(genre_name))

    return {
        "ok": True,
        "mood": mood_name,
        "mood_source": "manual_argument",
        "mood_detail": f"직접 지정한 mood={mood_name}",
        "build_started_at": build_started_at,
        "segment_number": SEGMENT_NO,
        "seed_expected_dir": expected_seed_dir,
        "seed_pt_path": LAST_SEED_INFO.get("pt_path"),
        "seed_pt_name": LAST_SEED_INFO.get("pt_name"),
        "seed_chunk_index": LAST_SEED_INFO.get("chunk_index"),
        "seed_num_chunks_in_pt": LAST_SEED_INFO.get("num_chunks_in_pt"),
        "same_as_previous_seed": same_as_previous_seed,
        "build_seconds": elapsed,
        "mp3_path": str(mp3_path),
    }


print("모델 로드 완료")
print(f"사용 가능한 mood: {', '.join(MOOD_MAP)}")
print(f"기본 생성 mood: {DEFAULT_START_MOOD}")


## 6. MP3 생성하기
원하는 mood 하나를 직접 지정해서 만들거나, 마지막 줄처럼 모든 mood를 일괄 생성할 수 있다.


In [ ]:
def _show_mp3_status(info):
    """생성된 MP3의 핵심 상태와 저장 경로를 출력한다."""
    print("\n" + "=" * 76)
    print(f"[DONE] segment={info['segment_number']} | mood={info['mood']}")
    print(f"[MOOD] source={info['mood_source']} | {info.get('mood_detail', '')}")
    print(
        f"[SEED] expected dir={info.get('seed_expected_dir')}\n"
        f"       actual={info.get('seed_pt_name')} / chunk={info.get('seed_chunk_index')} "
        f"(same as previous={info.get('same_as_previous_seed')})"
    )
    print(f"[BUILD] {info['build_seconds']}s")
    print(f"[MP3] {info['mp3_path']}")
    print("=" * 76)


def display_mp3_preview(mp3_path):
    """자동재생 없이 미리듣기 플레이어만 표시한다."""
    display(Audio(filename=str(mp3_path), autoplay=False, embed=True))


def generate_single_mp3(mood_name=DEFAULT_START_MOOD, preview=True):
    """지정한 mood로 MP3 하나를 생성하고 저장 경로를 보여준다."""
    info = build_one_mp3(mood_name=mood_name)
    _show_mp3_status(info)
    if preview:
        display_mp3_preview(info["mp3_path"])
    return info


def generate_all_moods_mp3(moods=None, preview=False):
    """모든 mood 또는 지정한 mood 목록을 순서대로 생성한다."""
    target_moods = list(MOOD_MAP) if moods is None else [_normalize_mood(mood) for mood in moods]
    results = []

    for index, mood_name in enumerate(target_moods, start=1):
        print(f"\n[{index}/{len(target_moods)}] {mood_name} 생성 시작")
        info = generate_single_mp3(mood_name=mood_name, preview=preview)
        results.append(info)

    print("\n전체 생성 완료")
    for info in results:
        print(f"- {info['mood']}: {info['mp3_path']}")
    return results


# 원하는 mood 하나만 생성하고 싶을 때 사용
# mp3_info = generate_single_mp3(mood_name=DEFAULT_START_MOOD)
# mp3_info = generate_single_mp3(mood_name="sad")

# 모든 mood를 일괄 생성
all_mp3_infos = generate_all_moods_mp3()
